# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines the March 2026 development contract for **Lane 2 — Refresh / Content Opportunity Scoring** and verifies it against the real warehouse. It deliberately does not use the June `_sample` table.

## 1. Data contract — five answers

**What one row represents — two distinct grains:** The **raw warehouse grain** is one record per `client_hash_id × content_hash_id × report_date`; one raw row represents one page for one date. After aggregating March feature and outcome windows, the **analytical ML grain** is one record per pseudonymized `client_hash_id × content_hash_id` pair for March 2026. The analytical dataframe therefore must contain one row per client/page pair.

**Warehouse tables:** `fact_content_daily_performance` is used for both feature and label windows. `dim_content` and `dim_clients` are supporting context tables, but this notebook does not need them for its March page-level calculations.

**Time window:** Development uses `month=2026-03`. Features come from March 1–15, while the proxy outcome comes from March 16–31. This separates information available at the decision point from later observed behavior.

**Predicted/ranked result:** The model ranks pages by estimated probability that the proxy `is_declining_proxy` is 1. The proxy is 1 when late-half average daily GSC impressions are lower than early-half average daily GSC impressions; it is a historical proxy, not proof of future decline or refresh benefit.

**Deliberately excluded:** Private URLs, client names, raw queries, product decision flags, identifiers used as predictive features, and all June `_sample` data. Identifiers are kept only as unit-of-analysis context.

In [1]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

# Load a local secret without printing, hardcoding, or writing it.
token = None
env_path = next((base / '.env' for base in [Path.cwd(), *Path.cwd().parents] if (base / '.env').exists()), None)
if env_path:
    for line in env_path.read_text().splitlines():
        if line.startswith('HF_TOKEN='):
            token = line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))

if not token:
    token = os.getenv('HF_TOKEN')

print('Local Hugging Face authentication is configured:', bool(token))

march_url = hf_hub_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet',
    token=token,
)

con = duckdb.connect()
print('Development month: 2026-03')
print('March parquet cached from the warehouse: yes')
print('Sample table excluded:', True)


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local Hugging Face authentication is configured: True


Development month: 2026-03
March parquet cached from the warehouse: yes
Sample table excluded: True


## 2. Fields and March feature frame

Field buckets:

The analytical feature dataframe retains `client_hash_id` and `content_hash_id` as context columns only; they are not among the five predictive features. This allows a uniqueness check on the analytical client/page grain.

- **Feature:** the five values listed below, all from March 1–15.
- **Label/proxy:** `is_declining_proxy`, derived only from March 16–31 average daily GSC impressions.
- **Context:** pseudonymized `client_hash_id` and `content_hash_id` for page grain and grouping only.
- **Excluded:** future-window impressions as a predictive feature; June `_sample` rows; raw queries; private URLs and names; and any product decision flag.

The feature frame below has five features at decision time:

1. `early_impressions`: GSC impressions summed over March 1–15, after `gsc_data_available IS TRUE`; knowable when the decision window closes.
2. `early_clicks`: GSC clicks summed over March 1–15 under the same availability rule; knowable then.
3. `weighted_avg_position`: impression-weighted GSC average position for March 1–15, available only from measured search rows.
4. `active_search_days`: number of distinct March 1–15 dates marked with GSC availability; activity is measurable before scoring.
5. `early_ctr`: early clicks divided by early impressions as a percentage (`100 × clicks / impressions`); knowable from the early feature window.

This feature query returns 30,000 real page examples for the small experiment. Its use of the future window is limited to creating the label and is removed before the final honest feature frame.

In [2]:
feature_query = '''
WITH page_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
        SUM(CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_avg_position * gsc_impressions ELSE 0 END)
          / NULLIF(SUM(CASE WHEN report_date <= DATE '2026-03-15'
                            THEN gsc_impressions ELSE 0 END), 0) AS weighted_avg_position,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15'
                            AND gsc_data_available IS TRUE THEN report_date END) AS active_search_days,
        SUM(CASE WHEN report_date > DATE '2026-03-15'
                 THEN gsc_impressions ELSE 0 END) AS late_impressions
    FROM read_parquet(?)
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
), feature_label AS (
    SELECT *,
           CASE WHEN early_impressions > 0
                THEN 100.0 * early_clicks / early_impressions ELSE NULL END AS early_ctr,
           CASE WHEN late_impressions / 16.0 < early_impressions / 15.0
                THEN 1 ELSE 0 END AS is_declining_proxy
    FROM page_windows
    WHERE early_impressions > 0 AND late_impressions >= 0
)
SELECT
    client_hash_id,
    content_hash_id,
    early_impressions,
    early_clicks,
    weighted_avg_position,
    active_search_days,
    early_ctr,
    late_impressions / 16.0 AS leaked_future_impressions_per_day,
    is_declining_proxy
FROM feature_label
WHERE weighted_avg_position IS NOT NULL AND early_ctr IS NOT NULL
ORDER BY hash(client_hash_id || content_hash_id)
LIMIT 30000
'''

feature_data = con.execute(feature_query, [march_url]).fetchdf()
feature_columns = [
    'early_impressions', 'early_clicks', 'weighted_avg_position',
    'active_search_days', 'early_ctr'
]
final_feature_frame = feature_data[['client_hash_id', 'content_hash_id'] + feature_columns].copy()
print('Feature dataframe shape:', feature_data.shape)
print('Target distribution:')
print(feature_data['is_declining_proxy'].value_counts().sort_index().rename(index={0: 'not declining proxy', 1: 'declining proxy'}).to_string())
unique_pair_count = final_feature_frame[['client_hash_id', 'content_hash_id']].drop_duplicates().shape[0]
pair_count_matches_rows = len(final_feature_frame) == unique_pair_count
print('Unique client/page pairs:', unique_pair_count)
print('Final dataframe row count matches unique pairs:', pair_count_matches_rows)
assert pair_count_matches_rows
print('Final honest feature frame shape:', final_feature_frame.shape)
print(final_feature_frame.head(10).round(4).to_string(index=False))

Feature dataframe shape: (30000, 9)
Target distribution:
is_declining_proxy
not declining proxy    15004
declining proxy        14996
Unique client/page pairs: 30000
Final dataframe row count matches unique pairs: True
Final honest feature frame shape: (30000, 7)
         client_hash_id          content_hash_id  early_impressions  early_clicks  weighted_avg_position  active_search_days  early_ctr
client_23a62021009f63c4 content_f5785c484503a6e1              154.0           0.0                19.0974                  15     0.0000
client_62f4a7e64f5e0096 content_378cab59bb132b3b               21.0           0.0                72.6667                  11     0.0000
client_a80fca3f171ed1de content_d216247d6d8f6d39                7.0           0.0                 4.0000                   6     0.0000
client_3f0ce4d44fe94f3d content_04519ccc2cf78e25              298.0           0.0                41.5369                  15     0.0000
client_3ffa76342f366962 content_7899ed5075ab5665        

## 3. March verification — exactly three small queries

**Raw warehouse grain vs analytical grain:** the warehouse grain is one raw record per client/page/report-date. The analytical ML grain is one aggregated record per client/page pair for March. These three queries verify the raw warehouse grain, March row/date window, and availability; the separate Python pair check below verifies the analytical grain. Each availability check uses `IS TRUE` rather than `= TRUE`, so unknown/null availability is not silently counted or excluded incorrectly.

In [3]:
verification_queries = {
    'grain_duplicate_check': '''
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
FROM read_parquet(?)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
''',
    'march_slice_count_and_span': '''
SELECT COUNT(*) AS march_rows,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date
FROM read_parquet(?)
''',
    'gsc_availability_count_is_true': '''
SELECT COUNT(*) AS available_rows
FROM read_parquet(?)
WHERE gsc_data_available IS TRUE
'''
}

grain_rows = con.execute(verification_queries['grain_duplicate_check'], [march_url]).fetchall()
slice_row = con.execute(verification_queries['march_slice_count_and_span'], [march_url]).fetchone()
available_row = con.execute(verification_queries['gsc_availability_count_is_true'], [march_url]).fetchone()

print('1. Grain duplicate rows (expected []):', grain_rows)
print('2. March slice: rows =', slice_row[0], '; span =', str(slice_row[1]), 'to', str(slice_row[2]))
print('3. gsc_data_available IS TRUE: available rows =', available_row[0])

import datetime
assert len(grain_rows) == 0
assert slice_row == (9841378, datetime.date(2026,3,1), datetime.date(2026,3,31))
assert available_row[0] == 3611061
print('Verification assertions passed:', True)

1. Grain duplicate rows (expected []): []
2. March slice: rows = 9841378 ; span = 2026-03-01 to 2026-03-31
3. gsc_data_available IS TRUE: available rows = 3611061
Verification assertions passed: True


## 4. Data limit and leakage experiment

**One limitation:** March 2026 is a mid-panel slice, and GSC availability is not complete: the March availability query shows that only 3,611,061 of 9,841,378 rows have `gsc_data_available IS TRUE`. Results describe this available search panel and should not be generalized to every page or to pages outside usable search history.

The experiment below deliberately adds one future-derived feature, `leaked_future_impressions_per_day`, and trains a small logistic model. It reports ROC-AUC. ROC-AUC measures how well the model ranks positive (declining-proxy) examples above negative examples; 0.50 is chance level and 1.00 is perfect ranking.

In [4]:
X_honest = feature_data[feature_columns].to_numpy(dtype=float)
y = feature_data['is_declining_proxy'].to_numpy(dtype=float)
X_leaked = feature_data[feature_columns + ['leaked_future_impressions_per_day']].to_numpy(dtype=float)

def logistic_roc_auc(X, y, random_seed=202603):
    rng = np.random.default_rng(random_seed)
    indices = rng.permutation(len(y))
    split = int(0.80 * len(y))
    train_idx, test_idx = indices[:split], indices[split:]

    mean = X[train_idx].mean(axis=0)
    std = X[train_idx].std(axis=0)
    std[std == 0] = 1.0
    train_X = (X[train_idx] - mean) / std
    test_X = (X[test_idx] - mean) / std
    train_X = np.column_stack([np.ones(len(train_X)), train_X])
    test_X = np.column_stack([np.ones(len(test_X)), test_X])

    weights = np.zeros(train_X.shape[1])
    train_y = y[train_idx]
    learning_rate = 0.1
    for _ in range(800):
        logits = np.clip(train_X @ weights, -35.0, 35.0)
        probabilities = 1.0 / (1.0 + np.exp(-logits))
        weights -= learning_rate * (train_X.T @ (probabilities - train_y)) / len(train_y)

    scores = test_X @ weights
    test_y = y[test_idx]
    order = np.argsort(scores, kind='mergesort')
    ordered_scores = scores[order]
    ordered_labels = test_y[order]

    ranks = np.empty(len(ordered_scores))
    i = 0
    while i < len(ordered_scores):
        j = i + 1
        while j < len(ordered_scores) and ordered_scores[j] == ordered_scores[i]:
            j += 1
        ranks[i:j] = (i + 1 + j) / 2.0
        i = j

    positive_count = ordered_labels.sum()
    negative_count = len(ordered_labels) - positive_count
    positive_rank_sum = sum(ranks[k] for k, label in enumerate(ordered_labels) if label == 1)
    return (positive_rank_sum - positive_count * (positive_count + 1) / 2.0) / (positive_count * negative_count)

honest_score = logistic_roc_auc(X_honest, y)
leaked_score = logistic_roc_auc(X_leaked, y)

print('Metric: ROC-AUC (80% deterministic train / 20% deterministic holdout)')
print('Honest score:', round(float(honest_score), 6))
print('Leaked score:', round(float(leaked_score), 6))
print('Score improvement with one leaked future feature:', round(float(leaked_score - honest_score), 6))

final_feature_frame = feature_data[['client_hash_id', 'content_hash_id'] + feature_columns].copy()
unique_pair_count = final_feature_frame[['client_hash_id', 'content_hash_id']].drop_duplicates().shape[0]
pair_count_matches_rows = len(final_feature_frame) == unique_pair_count
print('Unique analytical client/page pairs:', unique_pair_count)
print('Analytical row count matches unique pairs:', pair_count_matches_rows)
assert pair_count_matches_rows
assert 'leaked_future_impressions_per_day' not in final_feature_frame.columns
print('Leaked feature absent from final feature frame:', True)
print('Predictive feature count:', len(feature_columns))

Metric: ROC-AUC (80% deterministic train / 20% deterministic holdout)
Honest score: 0.575516
Leaked score: 0.801769
Score improvement with one leaked future feature: 0.226253
Unique analytical client/page pairs: 30000
Analytical row count matches unique pairs: True
Leaked feature absent from final feature frame: True
Predictive feature count: 5


## Self-check

Complete only the checked statements that are true:

- [x] Five plain-language data-contract answers are included.
- [x] Exactly three March verification queries are included: grain, slice/span, and `IS TRUE` availability.
- [x] The unit check returned no duplicate page-day grain rows.
- [x] The analytical dataframe has one unique client/page pair per row, and its row count equals the unique-pair count.
- [x] The feature frame has five features or fewer and explains when each is available.
- [x] One deliberately label/future-derived feature was added for the leakage experiment.
- [x] The honest score and leaked score use ROC-AUC on the same held-out sample.
- [x] The leaked feature was removed from the final feature frame.
- [x] One real limitation is stated.
- [x] No private URL, client name, raw query, token value, June `_sample` development, or product decision flag is displayed.
- [x] The notebook is executed top to bottom with no code-cell errors.
- [ ] Committed to the repository: intentionally not done; the user asked not to commit or push.

In [5]:
self_check = {
    'five_contract_answers': True,
    'three_verification_queries': len(verification_queries) == 3,
    'march_slice_actual': slice_row[0] == 9841378 and str(slice_row[1]) == '2026-03-01' and str(slice_row[2]) == '2026-03-31',
    'grain_holds': len(grain_rows) == 0,
    'availability_is_true_used': 'gsc_data_available IS TRUE' in verification_queries['gsc_availability_count_is_true'],
    'feature_count_at_most_five': len(feature_columns) <= 5,
    'leaked_feature_removed': 'leaked_future_impressions_per_day' not in final_feature_frame.columns,
    'analytical_pairs_unique': pair_count_matches_rows,
    'analytical_row_count_matches_unique_pairs': len(final_feature_frame) == unique_pair_count,
    'honest_score_reported': honest_score is not None,
    'leaked_score_reported': leaked_score is not None,
    'no_june_sample_development': 'fact_content_daily_performance_sample' not in march_url,
    'not_committed': True,
}

for check, passed in self_check.items():
    print(f'{check}: {"PASS" if passed else "FAIL"}')

assert all(self_check.values())
print('SELF-CHECK:', 'PASS')

five_contract_answers: PASS
three_verification_queries: PASS
march_slice_actual: PASS
grain_holds: PASS
availability_is_true_used: PASS
feature_count_at_most_five: PASS
leaked_feature_removed: PASS
analytical_pairs_unique: PASS
analytical_row_count_matches_unique_pairs: PASS
honest_score_reported: PASS
leaked_score_reported: PASS
no_june_sample_development: PASS
not_committed: PASS
SELF-CHECK: PASS
